# TF-IDF Baselines
Logistic Regression și SVM pe vectori TF-IDF. Reprezintă limita de jos a performanței — nu există nicio înțelegere contextuală, doar statistici de co-apariție a cuvintelor.

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
import joblib


## Configurare

In [ ]:
BASE      = os.path.expanduser("~/project/data")
MODEL_DIR = os.path.expanduser("~/project/model")
os.makedirs(MODEL_DIR, exist_ok=True)

LABEL_MAP = {"safe": 0, "potentially unsafe": 1, "unsafe": 2}


## Încărcare date

In [ ]:
def load_data():
    df_unsafe      = pd.read_json(f"{BASE}/Train/train_unsafe.jsonl", lines=True)
    df_safe        = pd.read_json(f"{BASE}/Train/train_safe.jsonl", lines=True)
    df_potentially = pd.read_json(f"{BASE}/Train/train_potentially_unsafe.jsonl", lines=True)
    df_full        = pd.concat([df_unsafe, df_safe, df_potentially], ignore_index=True)

    train_df, val_df = train_test_split(
        df_full, test_size=0.10, random_state=42,
        stratify=df_full['label'].str.strip().str.lower()
    )

    dv_unsafe      = pd.read_json(f"{BASE}/Validation/valid_unsafe.jsonl", lines=True)
    dv_safe        = pd.read_json(f"{BASE}/Validation/valid_safe.jsonl", lines=True)
    dv_potentially = pd.read_json(f"{BASE}/Validation/valid_potentially_unsafe.jsonl", lines=True)
    test_df        = pd.concat([dv_unsafe, dv_safe, dv_potentially], ignore_index=True)

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    return train_df, val_df, test_df

train_df, val_df, test_df = load_data()


In [ ]:
def make_text(row):
    return f"Query: {row.get('query', '')} Reasoning Trace: {row.get('reasoning_trace', '')}"

def get_labels(df):
    return df['label'].str.strip().str.lower().map(LABEL_MAP).fillna(0).astype(int).tolist()


## TF-IDF + Logistic Regression

In [ ]:
X_train = train_df.apply(make_text, axis=1).tolist()
X_test  = test_df.apply(make_text, axis=1).tolist()
y_train = get_labels(train_df)
y_test  = get_labels(test_df)

pipeline_lr = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50_000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=2,
    )),
    ('clf', LogisticRegression(
        C=1.0,
        max_iter=1000,
        class_weight='balanced',
        solver='lbfgs',
        multi_class='multinomial',
        n_jobs=-1,
    )),
])

print("Antrenare TF-IDF + LR ...")
pipeline_lr.fit(X_train, y_train)
joblib.dump(pipeline_lr, os.path.join(MODEL_DIR, "tfidf_lr.joblib"))


In [ ]:
preds_lr = pipeline_lr.predict(X_test)
acc_lr   = accuracy_score(y_test, preds_lr)
mf1_lr   = f1_score(y_test, preds_lr, average='macro', zero_division=0)

print(f"Accuracy: {acc_lr * 100:.2f}% | Macro-F1: {mf1_lr:.4f}")
print(classification_report(y_test, preds_lr,
      labels=[0, 1, 2],
      target_names=['safe', 'potentially_unsafe', 'unsafe'],
      zero_division=0))


## TF-IDF + SVM (LinearSVC)
`CalibratedClassifierCV` adaugă probabilități la `LinearSVC` prin cross-validation intern.

In [ ]:
pipeline_svm = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=50_000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=2,
    )),
    ('clf', CalibratedClassifierCV(
        LinearSVC(C=1.0, max_iter=2000, class_weight='balanced'),
        cv=3
    )),
])

print("Antrenare TF-IDF + SVM ...")
pipeline_svm.fit(X_train, y_train)
joblib.dump(pipeline_svm, os.path.join(MODEL_DIR, "tfidf_svm.joblib"))


In [ ]:
preds_svm = pipeline_svm.predict(X_test)
acc_svm   = accuracy_score(y_test, preds_svm)
mf1_svm   = f1_score(y_test, preds_svm, average='macro', zero_division=0)

print(f"Accuracy: {acc_svm * 100:.2f}% | Macro-F1: {mf1_svm:.4f}")
print(classification_report(y_test, preds_svm,
      labels=[0, 1, 2],
      target_names=['safe', 'potentially_unsafe', 'unsafe'],
      zero_division=0))


## Sumar

In [ ]:
print(f"{'Model':<20} | {'Accuracy':>9} | {'Macro-F1':>9}")
print("-" * 45)
print(f"{'TF-IDF + LR':<20} | {acc_lr*100:>8.2f}% | {mf1_lr:>9.4f}")
print(f"{'TF-IDF + SVM':<20} | {acc_svm*100:>8.2f}% | {mf1_svm:>9.4f}")
